In [ ]:
import os
import io
import re
import json
import pandas as pd
from pathlib import Path

In [ ]:
with open('responses/photo_3433@01-04-2026_11-37-10.json', 'r', encoding='utf8') as jf:
    d = json.loads(jf.read())

In [ ]:
json.loads(d)

In [ ]:
with open('responses/photo_3439@04-04-2026_12-12-28.json', 'r', encoding='utf8') as jf:
    contents = jf.read()
    d = json.loads(contents)

In [ ]:
mdtable = d['pages'][0]['markdown'].replace('|n|', '| \n |')

In [ ]:
with open('test.md', 'w+', encoding='utf8') as f:
    f.write(mdtable)

In [ ]:
def read_json(path):
    with open(path, 'r', encoding='utf8') as jf:
        json_string = json.loads(jf.read())
        if isinstance(json_string, str):
            return json.loads(json_string)
        else:
            return json_string

In [ ]:
# pipeline
def pipeline(f):
    data = read_json(f)
    md_content = data['pages'][0]['markdown'].rsplit('|', 1)[0]
    
    if 'РАУНД 1' not in md_content:
        return

    md_content = md_content.rsplit('| --- |', 1)[-1]
    md_content = re.sub(r'\s*\([^)]*\)\s*\n?', '', md_content)
    
    try:
        df = pd.read_table(
            io.StringIO(md_content), 
            sep="|", 
            header=0,
            skipinitialspace=True
        ).dropna(axis=1, how='all').iloc[1:]
    except Exception:
        with open(f'{Path(f).stem}_error.md', 'w+', encoding='utf8') as f:
            f.write(md_content)
    
    if md_content and df.empty:
        md_content = re.sub(r'\bn\b', r'\n', md_content)
        df = pd.read_table(
            io.StringIO(md_content), 
            sep="|", 
            header=0,
            skipinitialspace=True
        ).dropna(axis=1, how='all').iloc[1:]
        if df.empty:
            print(f + ' EMPTY DATAFRAME')
    
    df.to_csv(f'csvs/{Path(f).stem}.csv', index=False)
    
    with open(f'mds/{Path(f).stem}.md', 'w+', encoding='utf8') as mdf:
        mdf.write(md_content)

In [ ]:
pipeline('responses/photo_3439@04-04-2026_12-12-28.json')

In [ ]:
for p in os.listdir('responses'):
    pipeline(f'responses/{p}')